In [5]:
pip install pandas numpy matplotlib seaborn scikit-learn

In [6]:
from google.colab import files

# Upload the ratings.dat and movies.dat files
uploaded = files.upload()

Saving movies.dat to movies (4).dat
Saving ratings.dat to ratings (3).dat


In [7]:
import pandas as pd

#  loading the movies file with 'ISO-8859-1' encoding
movies = pd.read_csv("movies.dat", sep="::", engine="python", header=None, names=["movieId", "title", "genres"], encoding="ISO-8859-1")

# Check the data
print(movies.head())

   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [8]:
# Load ratings.dat with 'ISO-8859-1' encoding
ratings = pd.read_csv("ratings.dat", sep="::", engine="python", header=None, names=["userId", "movieId", "rating", "timestamp"], encoding="ISO-8859-1")

# Check the data
print(ratings.head())

   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291


In [9]:
# Merge the ratings and movies datasets on the 'movieId' column
merged_data = pd.merge(ratings, movies, on="movieId")

# Check the first few rows of the merged dataset
print(merged_data.head())

   userId  movieId  rating  timestamp                                   title  \
0       1     1193       5  978300760  One Flew Over the Cuckoo's Nest (1975)   
1       1      661       3  978302109        James and the Giant Peach (1996)   
2       1      914       3  978301968                     My Fair Lady (1964)   
3       1     3408       4  978300275                  Erin Brockovich (2000)   
4       1     2355       5  978824291                    Bug's Life, A (1998)   

                         genres  
0                         Drama  
1  Animation|Children's|Musical  
2               Musical|Romance  
3                         Drama  
4   Animation|Children's|Comedy  


In [10]:
# Check basic info about the merged data
print(merged_data.info())

# Show summary statistics
print(merged_data.describe())

# Check number of unique users and movies
print(f"Unique users: {merged_data['userId'].nunique()}")
print(f"Unique movies: {merged_data['movieId'].nunique()}")

# Show the most popular movies based on the number of ratings
movie_counts = merged_data['title'].value_counts()
print(movie_counts.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 6 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   userId     1000209 non-null  int64 
 1   movieId    1000209 non-null  int64 
 2   rating     1000209 non-null  int64 
 3   timestamp  1000209 non-null  int64 
 4   title      1000209 non-null  object
 5   genres     1000209 non-null  object
dtypes: int64(4), object(2)
memory usage: 45.8+ MB
None
             userId       movieId        rating     timestamp
count  1.000209e+06  1.000209e+06  1.000209e+06  1.000209e+06
mean   3.024512e+03  1.865540e+03  3.581564e+00  9.722437e+08
std    1.728413e+03  1.096041e+03  1.117102e+00  1.215256e+07
min    1.000000e+00  1.000000e+00  1.000000e+00  9.567039e+08
25%    1.506000e+03  1.030000e+03  3.000000e+00  9.653026e+08
50%    3.070000e+03  1.835000e+03  4.000000e+00  9.730180e+08
75%    4.476000e+03  2.770000e+03  4.000000e+00  9.752209e+08
max    6

In [11]:
!pip install numpy==1.24.3
!pip install scikit-surprise

  Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.21.2 requires numpy>=1.25.0, but you have numpy 1.24.3 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.24.3 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.3 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.24.3 which is incompatible.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.24.3 which is incompatible.
jaxlib 0.5.1 requires numpy

In [12]:
from surprise import Dataset, Reader
from surprise import SVD, accuracy
from surprise.model_selection import train_test_split

# Prepare the data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(merged_data[['userId', 'movieId', 'rating']], reader)

# Split the data into training and test sets
trainset, testset = train_test_split(data, test_size=0.2)

# Train the SVD model
model = SVD()
model.fit(trainset)

# Predict and evaluate
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"RMSE: {rmse}")


RMSE: 0.8707
RMSE: 0.8707062956881827


In [13]:
user_id = 1  # User ID
movie_id = 100  # Movie ID (e.g., the ID for "Star Wars: Episode IV")

# Predict the rating
prediction = model.predict(user_id, movie_id)
print(f"Predicted rating for user {user_id} on movie {movie_id}: {prediction.est}")


Predicted rating for user 1 on movie 100: 3.450213015151723


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create a TF-IDF Vectorizer for the 'genres' column
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the genres column
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Compute cosine similarity between movies based on genres
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a mapping of movieId to title
movie_indices = pd.Series(movies.index, index=movies['title']).to_dict()

# Define a function to get recommendations based on movie title
def content_based_recommender(title, cosine_sim=cosine_sim, movie_indices=movie_indices): # Pass movie_indices as an argument
    # Get the index of the movie that matches the title
    idx = movie_indices[title]

    # Get the pairwise similarity scores for that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the top 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices_local = [i[0] for i in sim_scores] # Rename local variable to avoid shadowing

    # Return the top 10 most similar movies
    return movies['title'].iloc[movie_indices_local]

# Test the content-based recommender
print(content_based_recommender("Star Wars: Episode IV - A New Hope (1977)"))

2036                                         Tron (1982)
2559    Star Wars: Episode I - The Phantom Menace (1999)
2104          Navigator: A Mediaeval Odyssey, The (1988)
2899                                 Time Bandits (1981)
1006                 20,000 Leagues Under the Sea (1954)
1698                                     Star Kid (1997)
2024                                 Return to Oz (1985)
1952                                         Dune (1984)
1985                     Honey, I Shrunk the Kids (1989)
171                                   Judge Dredd (1995)
Name: title, dtype: object


In [15]:
# Function to get hybrid recommendations
def hybrid_recommender(user_id, movie_id, top_n=10):
    # Get collaborative recommendations (from SVD model)
    collab_pred = model.predict(user_id, movie_id)

    # Get content-based recommendations
    movie_title = movies[movies['movieId'] == movie_id]['title'].values[0]
    content_recs = content_based_recommender(movie_title)

    # Combine the recommendations
    print(f"Collaborative prediction: {collab_pred.est}")
    print(f"Top 10 Content-Based Recommendations:")
    for movie in content_recs:
        print(movie)

# Example usage: hybrid recommender for user 1 and movie 100
hybrid_recommender(8, 108)


Collaborative prediction: 3.7134776973953434
Top 10 Content-Based Recommendations:
Nico Icon (1995)
Heidi Fleiss: Hollywood Madam (1995)
Catwalk (1995)
Anne Frank Remembered (1995)
Jupiter's Wife (1994)
Sonic Outlaws (1995)
From the Journals of Jean Seberg (1995)
Man of the Year (1995)
Crumb (1994)
Show, The (1995)


In [16]:
# Create a mapping of movieId to title
movie_indices = pd.Series(movies.index, index=movies['title']).to_dict()

# Define a function to get recommendations based on movie title
def content_based_recommender(title, cosine_sim=cosine_sim, movie_indices=movie_indices):
    # Get the index of the movie that matches the title
    idx = movie_indices.get(title)  # Use .get to avoid KeyError

    if idx is None:
        return "Movie not found in dataset"

    # Get the pairwise similarity scores for that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the top 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices_local = [i[0] for i in sim_scores]  # Renamed to avoid shadowing

    # Return the top 10 most similar movies
    return movies['title'].iloc[movie_indices_local]  # Use local variable here

# Test the content-based recommender
print(content_based_recommender("Star Wars: Episode IV - A New Hope (1977)"))


2036                                         Tron (1982)
2559    Star Wars: Episode I - The Phantom Menace (1999)
2104          Navigator: A Mediaeval Odyssey, The (1988)
2899                                 Time Bandits (1981)
1006                 20,000 Leagues Under the Sea (1954)
1698                                     Star Kid (1997)
2024                                 Return to Oz (1985)
1952                                         Dune (1984)
1985                     Honey, I Shrunk the Kids (1989)
171                                   Judge Dredd (1995)
Name: title, dtype: object


**Collaborative Filtering**

recommend movies based on user behavior (ratings they give).
There are two common types:

User-User Collaborative Filtering

Item-Item Collaborative Filtering

In [2]:
!pip install numpy==1.26.4 --force-reinstall


  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [17]:
# Step 2: Import libraries
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

# Step 3: Prepare the data
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(merged_data[['userId', 'movieId', 'rating']], reader)

# Step 4: Train/Test Split
trainset, testset = train_test_split(data, test_size=0.2)

# Step 5: Build and Train the SVD model
model = SVD()
model.fit(trainset)

# Step 6: Make Predictions
predictions = model.test(testset)

# Step 7: Evaluate the Model
rmse = accuracy.rmse(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")


RMSE: 0.8755
Root Mean Squared Error (RMSE): 0.8754814576375041


In [18]:
# Let's recommend top 5 movies for a user
user_id = 1  # Example: for user 1

# Get all movies the user hasn't rated yet
user_movies = merged_data[merged_data['userId'] == user_id]['movieId'].unique()
all_movies = merged_data['movieId'].unique()
movies_to_predict = list(set(all_movies) - set(user_movies))

# Predict ratings
predictions = [model.predict(user_id, movie_id) for movie_id in movies_to_predict]

# Sort predictions
top_predictions = sorted(predictions, key=lambda x: x.est, reverse=True)[:5]

# Show top recommended movie IDs
top_movie_ids = [pred.iid for pred in top_predictions]
print("Top 5 recommended movie IDs for User 1:", top_movie_ids)


Top 5 recommended movie IDs for User 1: [858, 1252, 668, 912, 1294]


**Content-Based Filtering**

In [20]:
# Import libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Instead of loading from 'movies.csv', use the existing 'movies' DataFrame
# movies = pd.read_csv('movies.csv')  # Path depends on your dataset

# Check the first rows (optional, to verify it's the correct data)
print(movies.head())

# Fill missing genres with empty string (if necessary)
movies['genres'] = movies['genres'].fillna('')

# Use TF-IDF to convert genres into feature vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Reset index of your DataFrame and construct reverse mapping
movies = movies.reset_index()
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# Recommendation function
def recommend_content_based(title, cosine_sim=cosine_sim):
    idx = indices[title]

    # Get the similarity scores for all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get scores of 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 similar movies
    return movies['title'].iloc[movie_indices]

# Example usage:
print(recommend_content_based('Toy Story (1995)'))

   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
1050            Aladdin and the King of Thieves (1996)
2072                          American Tail, An (1986)
2073        American Tail: Fievel Goes West, An (1991)
2285                         Rugrats Movie, The (1998)
2286                              Bug's Life, A (1998)
3045                                Toy Story 2 (1999)
3542                             Saludos Amigos (1943)
3682                                Chicken Run (2000)
3685    Adventures of Rocky and Bullwinkle, The (2000)
12                                        B

**Hybrid Recommender**

In [21]:
# Import necessary libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# Assume 'merged_data' and 'movies' DataFrames already exist
# Assume 'model' (SVD) is already trained

# Content-Based setup (you already did)
movies['genres'] = movies['genres'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
movies = movies.reset_index()
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [22]:
# Hybrid recommendation function
def hybrid_recommend(user_id, title, model, cosine_sim=cosine_sim):
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get scores of 25 most similar movies
    sim_scores = sim_scores[1:26]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Movies titles
    similar_movies = movies['movieId'].iloc[movie_indices]

    # Predict ratings using the Collaborative Filtering model
    cf_predictions = []
    for movie_id in similar_movies:
        pred = model.predict(user_id, movie_id)
        cf_predictions.append((movie_id, pred.est))

    # Sort by predicted rating
    cf_predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 10 movieIds
    top_movie_ids = [movie_id for movie_id, _ in cf_predictions[:10]]

    # Return movie titles
    return movies[movies['movieId'].isin(top_movie_ids)]['title']

# Example usage:
print(hybrid_recommend(user_id=1, title='Toy Story (1995)', model=model))


592                                Pinocchio (1940)
612                          Aristocats, The (1970)
1010    Winnie the Pooh and the Blustery Day (1968)
1012                 Sword in the Stone, The (1963)
1838                                   Mulan (1998)
1949                                   Bambi (1942)
2016                          101 Dalmatians (1961)
2286                           Bug's Life, A (1998)
3045                             Toy Story 2 (1999)
3682                             Chicken Run (2000)
Name: title, dtype: object
